# CEG-WM HF-only threshold-fit GPU execution shard

This output-free Notebook only verifies a package-external bootstrap and invokes one explicitly selected HF-only threshold-fit GPU execution shard. It cannot approve tau, unlock confirmation data, run baselines, or create scientific claims.

In [ ]:
from google.colab import drive, files, userdata
from pathlib import Path
from hashlib import sha256
import json
import os
import subprocess
import sys
import tempfile

drive.mount('/content/drive')

In [ ]:
DRIVE_ROOT = Path('/content/drive/MyDrive/CEG-WM')
BOOTSTRAP_SOURCE = DRIVE_ROOT / 'bootstrap/experiment_execution_bootstrap.py'
PACKAGE_ZIP = DRIVE_ROOT / 'execution_packages/current/ceg_wm_hf_only_threshold_fit.zip'
DELIVERY_MANIFEST = DRIVE_ROOT / 'execution_packages/current/ceg_wm_hf_only_threshold_fit.zip.manifest.json'
EXPECTED_BOOTSTRAP_IDENTITY = 'ceg_wm_experiment_execution_bootstrap'
EXPECTED_BOOTSTRAP_SCHEMA_VERSION = '2'
EXPECTED_BOOTSTRAP_SHA256 = 'PASTE_INDEPENDENTLY_AUDITED_BOOTSTRAP_SHA256'
EXPECTED_ARCHIVE_SHA256 = 'PASTE_INDEPENDENTLY_AUDITED_ARCHIVE_SHA256'
EXPECTED_DELIVERY_MANIFEST_SHA256 = 'PASTE_INDEPENDENTLY_AUDITED_SIDECAR_SHA256'
EXPECTED_EMBEDDED_MANIFEST_SHA256 = 'PASTE_INDEPENDENTLY_AUDITED_EMBEDDED_MANIFEST_SHA256'
EXPECTED_REVISION = 'PASTE_INDEPENDENTLY_AUDITED_40_HEX_REVISION'
RUN_ID = 'PASTE_UNIQUE_RUN_ID'
SHARD_INDEX = 0
EPHEMERAL_ROOT = Path('/content/ceg_wm_hf_only_threshold_fit')
PERSISTENT_ROOT = DRIVE_ROOT / 'hf_only_threshold_fit_results'
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
os.environ['CEG_WM_ROOT_KEY'] = userdata.get('CEG_WM_ROOT_KEY')

In [ ]:
bootstrap_bytes = BOOTSTRAP_SOURCE.read_bytes()
assert sha256(bootstrap_bytes).hexdigest() == EXPECTED_BOOTSTRAP_SHA256
BOOTSTRAP_SNAPSHOT_ROOT = Path(tempfile.mkdtemp(prefix='ceg_wm_threshold_bootstrap_', dir='/content'))
BOOTSTRAP_SNAPSHOT = BOOTSTRAP_SNAPSHOT_ROOT / 'experiment_execution_bootstrap.py'
with BOOTSTRAP_SNAPSHOT.open('xb') as destination:
    destination.write(bootstrap_bytes)
assert sha256(BOOTSTRAP_SNAPSHOT.read_bytes()).hexdigest() == EXPECTED_BOOTSTRAP_SHA256

In [ ]:
command = [
    sys.executable, str(BOOTSTRAP_SNAPSHOT),
    '--package-zip', str(PACKAGE_ZIP),
    '--delivery-manifest-path', str(DELIVERY_MANIFEST),
    '--expected-archive-sha256', EXPECTED_ARCHIVE_SHA256,
    '--expected-delivery-manifest-sha256', EXPECTED_DELIVERY_MANIFEST_SHA256,
    '--expected-embedded-manifest-sha256', EXPECTED_EMBEDDED_MANIFEST_SHA256,
    '--expected-bootstrap-identity', EXPECTED_BOOTSTRAP_IDENTITY,
    '--expected-bootstrap-schema-version', EXPECTED_BOOTSTRAP_SCHEMA_VERSION,
    '--expected-bootstrap-sha256', EXPECTED_BOOTSTRAP_SHA256,
    '--expected-revision', EXPECTED_REVISION,
    '--ephemeral-root', str(EPHEMERAL_ROOT),
    '--persistent-root', str(PERSISTENT_ROOT),
    '--run-id', RUN_ID,
    '--shard-index', str(SHARD_INDEX),
]
completed = subprocess.run(command, check=False, capture_output=True, text=True)
print(completed.stdout)
if completed.stderr:
    print(completed.stderr)
result = json.loads(completed.stdout)
assert completed.returncode in (0, 3, 4)

In [ ]:
artifact_path = result.get('result_zip') or result.get('diagnostic_zip')
assert artifact_path
print(result['artifact_kind'], artifact_path)
files.download(artifact_path)